# ARC-AGI Solver — Evaluate

Runs inference (greedy / TTA / TTT) on a trained checkpoint stored in Google Drive.
Evaluation is leave-one-out over the original ARC training pairs for each task.

**Cell order:**
1. Check GPU → 2. Mount Drive → 3. Clone repo + ARC data → 4. Config → 5. Evaluate

**Modes:**
- `greedy` — single forward pass, argmax per cell
- `tta`    — N colour permutations → un-permute → majority vote per cell
- `ttt`    — fine-tune on task training pairs (M steps) then run TTA
- `all`    — run all three and print a comparison table

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — inference will be slow on CPU.')

In [ ]:
# ── Cell 2: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE     = '/content/drive/MyDrive/arc-agi'
DRIVE_CKPT_DIR = f'{DRIVE_BASE}/checkpoints'

print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')
ckpts = sorted(Path(DRIVE_CKPT_DIR).glob('*.pt'))
print(f'Available checkpoints ({len(ckpts)}):')
for p in ckpts:
    print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Cell 3: Clone repo and download ARC training data ───────────────────────
import os, io, urllib.request, zipfile
from pathlib import Path

# Clone / pull repo
REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
if not os.path.isdir(REPO):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git')
else:
    os.system(f'git -C {REPO} pull --ff-only')
os.chdir(REPO)
print(f'Working directory: {os.getcwd()}')
os.system('git log --oneline -3')

# ARC training data (evaluate.py reads from data/training/)
DATA_DIR = Path('data')
if (DATA_DIR / 'training').exists() and len(list((DATA_DIR / 'training').glob('*.json'))) >= 400:
    print(f"ARC data already present ({len(list((DATA_DIR/'training').glob('*.json')))} tasks).")
else:
    print('Downloading ARC-AGI dataset...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    ) as r:
        raw = r.read()
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        dest = DATA_DIR / 'training'
        dest.mkdir(exist_ok=True)
        for m in zf.namelist():
            if 'data/training/' in m and m.endswith('.json'):
                (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"training: {len(list((DATA_DIR/'training').glob('*.json')))} tasks")

In [ ]:
# ── Cell 4: Evaluation configuration ────────────────────────────────────────
# Set RUN_NAME to match the checkpoint you want to evaluate.
# The checkpoint is loaded from DRIVE_CKPT_DIR automatically.

from pathlib import Path

RUN_NAME  = 'C8_compartment_fill'   # ← matches the training run name
K_CONTEXT = 3                        # context pairs at inference time

# ── Inference options ─────────────────────────────────────────────────────────
EVAL_MODE  = 'all'   # 'greedy' | 'tta' | 'ttt' | 'all'
N_PERMS    = 20      # colour permutations for TTA
TTT_STEPS  = 50      # fine-tune steps per task for TTT
TTT_LR     = 1e-4    # TTT learning rate

# ── Resolve checkpoint path ───────────────────────────────────────────────────
ckpt_path = str(Path(DRIVE_CKPT_DIR) / f'transformer_c{RUN_NAME}_best.pt')
if Path(ckpt_path).exists():
    sz = Path(ckpt_path).stat().st_size / 1e6
    print(f'Checkpoint: {ckpt_path}  ({sz:.1f} MB)  ✓')
else:
    print(f'Checkpoint NOT FOUND: {ckpt_path}')
    print('Available:')
    for p in sorted(Path(DRIVE_CKPT_DIR).glob('*.pt')):
        print(f'  {p.name}')

In [ ]:
# ── Cell 5: Run evaluation ───────────────────────────────────────────────────
# Evaluates using leave-one-out over the original ARC training pairs:
#   for each training pair i: context = all other pairs, query = pair i
#
# Expected runtime on A100:
#   greedy: ~1 min   tta (20 perms): ~2 min   ttt (50 steps): ~3 min
import subprocess, sys

cmd = [
    sys.executable, '-u', 'scripts/evaluate.py',
    '--checkpoint', ckpt_path,
    '--mode',       EVAL_MODE,
    '--n-perms',    str(N_PERMS),
    '--ttt-steps',  str(TTT_STEPS),
    '--ttt-lr',     str(TTT_LR),
    '--k-context',  str(K_CONTEXT),
    '--verbose',
]

print('Mode:', EVAL_MODE)
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')